# CS570 Week 4: Lab Class - Joins & Window Functions

**Today's session:** Interactive coding together (not graded)

**Homework:** The separate Lab notebook (graded, due on Canvas)

---

## Rules for Today

1. **Don't run ahead** - we do this together
2. **Predict before running** - write down your prediction, THEN run
3. **Ask questions** - if you're confused, others are too

---

## Part 1: Setup (everyone together)

Run these cells. Wait for checkpoint before continuing.

In [1]:
import os
import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Ensure Spark workers use the same Python as this notebook
# (Fixes version mismatch errors when using virtual environments)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Create SparkSession with all available cores
spark = SparkSession.builder \
    .appName("CS570 Spotify Analysis") \
    .master("local[*]") \
    .getOrCreate()

# Get SparkContext from session (for RDD operations)
sc = spark.sparkContext

print(f"Spark version: {spark.version}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Default parallelism (cores): {sc.defaultParallelism}")

sc.setLogLevel("ERROR")  # Suppress warnings for cleaner output
print(f"\n✓ Spark is ready with {sc.defaultParallelism} cores!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/04 19:21:09 WARN Utils: Your hostname, News-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.16.9.72 instead (on interface en0)
26/02/04 19:21:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/04 19:21:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Python version: 3.11.14
Default parallelism (cores): 12

✓ Spark is ready with 12 cores!


In [2]:
# Load Spotify data
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv("spotify.csv")

row_count = df.count()
print(f"Rows loaded: {row_count:,}")

Rows loaded: 114,000


In [47]:
df.show(5)

+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|_c0|            track_id|             artists|          album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|
+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|  0|5SuOikwiRyPMVoIQD...|         Gen Hoshino|              Comedy|              Comedy|        73|     230666|   false|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|         1.01E-6|   0.358|  0.715| 87.917|            

### ✅ CHECKPOINT

**Call out your row count.** Everyone should have **114,000**.

If you don't, raise your hand.

---

## Part 2: Joins

### 2.1 Create a lookup table

In real data pipelines, you often have:
- A **fact table** (transactions, events, tracks)
- A **dimension/lookup table** (categories, users, genres)

Let's create a small genre lookup table:

In [3]:
# A small lookup table with extra info about some genres
genre_data = [
    ("pop", "Popular Music", "High"),
    ("rock", "Rock Music", "High"),
    ("jazz", "Jazz Music", "Medium"),
    ("classical", "Classical Music", "Medium"),
    ("fake_genre", "Doesn't Exist In Tracks", "Low")
]

genres = spark.createDataFrame(genre_data, ["genre", "full_name", "market_size"])
genres.show()

+----------+--------------------+-----------+
|     genre|           full_name|market_size|
+----------+--------------------+-----------+
|       pop|       Popular Music|       High|
|      rock|          Rock Music|       High|
|      jazz|          Jazz Music|     Medium|
| classical|     Classical Music|     Medium|
|fake_genre|Doesn't Exist In ...|        Low|
+----------+--------------------+-----------+



Notice:
- We have info for **pop, rock, jazz, classical** (these exist in tracks)
- We have **fake_genre** (does NOT exist in tracks)
- We're **missing** many genres that ARE in tracks (hip-hop, electronic, etc.)

---

### 2.2 Prediction: INNER JOIN

I'm about to run:
```python
df.join(...)
```

## ✋ STOP - PREDICT FIRST

**Write your predictions below before running the next cell:**

1. Will we get MORE rows, FEWER rows, or SAME rows as original (114,000)?
   
   → My prediction: _____________

2. Will "fake_genre" appear in the result?
   
   → My prediction: _____________

3. What about tracks with genre "hip-hop" (not in our lookup table)?
   
   → My prediction: _____________

In [48]:
# INNER JOIN
df.join(genres, on = df.track_genre == genres.genre, how = 'inner').show(5)

+-----+--------------------+--------------------+--------------------+-------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+-----+-------------+-----------+
|  _c0|            track_id|             artists|          album_name|         track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|genre|    full_name|market_size|
+-----+--------------------+--------------------+--------------------+-------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+-----+-------------+-----------+
|81000|0VjIjW4GlUZAMYd2v...|          The Weeknd|         After Hours|    Blinding Lights|        91|     200040|   false|       0.514|  0.

### 2.3 Prediction: LEFT JOIN

Now I'll do a LEFT join.

## ✋ STOP - PREDICT FIRST

1. Will we get MORE rows, FEWER rows, or SAME rows as original (114,000)?
   
   → My prediction: _____________

2. For tracks with genre "hip-hop" (not in lookup), what will the columns from the lookup table contain?
   
   → My prediction: _____________

In [49]:
# LEFT JOIN
df.join(genres, df.track_genre == genres.genre, 'left').filter(df.track_genre == 'hip-hop').show(5)

+-----+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+-----+---------+-----------+
|  _c0|            track_id|             artists|          album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|genre|full_name|market_size|
+-----+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+-----+---------+-----------+
|51000|1aL9518P5G72N92b4...|          AP Dhillon|         Summer High|         Summer High|        83|     177391|   false|        0.86| 0.541|  1| 

In [14]:
# Show rows where no match was found (NULLs in lookup columns)
df.filter(col("track_genre").isNull()).show()

+---+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|_c0|track_id|artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|track_genre|
+---+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
+---+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+



### 2.4 Data Quality Check: LEFT ANTI

We just used left join + filter for NULL to find non-matches. But there's a cleaner way.

**Question:** What if I *only* want the tracks that don't have a matching genre in the lookup?

This is `left_anti` - rows from left where key does **NOT** exist in right. No NULLs to filter, no extra columns added.

In [21]:
# LEFT ANTI - find tracks with no matching genre info
df.join(genres, df.track_genre == genres.genre, how = 'left_anti').select("track_name","Artists", "track_genre").show()

+--------------------+--------------------+-----------+
|          track_name|             Artists|track_genre|
+--------------------+--------------------+-----------+
|      カワキヲアメク|                美波|      anime|
|          シルエット|           KANA-BOON|      anime|
|           KICK BACK|       Kenshi Yonezu|      anime|
|             unravel|TK from Ling tosi...|      anime|
|  残酷な天使のテーゼ|      Yoko Takahashi|      anime|
|        Kaikai Kitan|                 Eve|      anime|
|        ピースサイン|       Kenshi Yonezu|      anime|
|               FEVER|             ENHYPEN|      anime|
|Shingeki Gt 20130...|     Hiroyuki Sawano|      anime|
|        ブルーバード|       Ikimonogakari|      anime|
|              人芝居|             渕上 舞|      anime|
|          FLY HIGH!!|   BURNOUT SYNDROMES|      anime|
|      crossing field|                LiSA|      anime|
|                  炎|                LiSA|      anime|
|       Black Catcher|         Vickeblanka|      anime|
|              紅蓮華|               

In [24]:
# Which genres are we missing info for?
df.join(genres, df.track_genre == genres.genre, how = 'left_anti').groupBy("track_genre").count().show()

+-----------------+-----+
|      track_genre|count|
+-----------------+-----+
|            anime| 1000|
|singer-songwriter| 1000|
|             folk| 1000|
|        hardstyle| 1000|
|      alternative| 1000|
|      death-metal| 1000|
|   detroit-techno| 1000|
|              idm| 1000|
|            k-pop| 1000|
|          j-dance| 1000|
|          ambient| 1000|
|           guitar| 1000|
|             goth| 1000|
|         cantopop| 1000|
|            blues| 1000|
|            study| 1000|
|            malay| 1000|
|        breakbeat| 1000|
|            dance| 1000|
|           groove| 1000|
+-----------------+-----+
only showing top 20 rows


### Summary: Join Types

| Type | Returns |
|------|--------|
| `inner` | Only rows that match both sides |
| `left` | All left rows + matching right (NULL if no match) |
| `left_anti` | Left rows where key NOT in right |
| `left_semi` | Left rows where key exists in right (no right columns) |

---

## Part 3: Window Functions

### The Problem

**Question: Find the most popular track in each genre.**

Try to use a `groupBy`?

In [25]:
#YOUR CODE
max_pop = df.groupby("track_genre").agg(max("popularity")).show()

+-----------------+---------------+
|      track_genre|max(popularity)|
+-----------------+---------------+
|            anime|             83|
|singer-songwriter|             90|
|             folk|             90|
|        hardstyle|             70|
|              pop|            100|
|      alternative|             93|
|      death-metal|             73|
|   detroit-techno|             58|
|              idm|             70|
|            k-pop|             88|
|          j-dance|             77|
|          ambient|             84|
|           guitar|             71|
|             goth|             67|
|         cantopop|             72|
|            blues|             84|
|            study|             55|
|            malay|             72|
|        breakbeat|             66|
|            dance|            100|
+-----------------+---------------+
only showing top 20 rows


### 3.1 Define the Window

Window functions give us a cleaner way: **add aggregate info without collapsing rows.**

A window has two parts:
- `partitionBy` - what are the groups? (like GROUP BY)
- `orderBy` - how to sort within each group?

In [29]:
# Define window: within each genre, order by popularity (highest first)
w = Window.partitionBy("track_genre").orderBy(desc("popularity"))

df.withColumn("rank", rank().over(w)).select("track_name","Artists", "track_genre", "popularity", "rank").show()

+--------------------+--------------------+-----------+----------+----+
|          track_name|             Artists|track_genre|popularity|rank|
+--------------------+--------------------+-----------+----------+----+
|     Sweater Weather|   The Neighbourhood|alternative|        93|   1|
|        Daddy Issues|   The Neighbourhood|alternative|        87|   2|
|            Miss You|Oliver Tree;Robin...|alternative|        87|   2|
|            Softcore|   The Neighbourhood|alternative|        86|   4|
|             abcdefu|               GAYLE|alternative|        86|   4|
|      Mr. Brightside|         The Killers|alternative|        86|   4|
|          In the End|         Linkin Park|alternative|        85|   7|
|               Creep|           Radiohead|alternative|        85|   7|
|   Seven Nation Army|   The White Stripes|alternative|        84|   9|
|      Feel Good Inc.|            Gorillaz|alternative|        84|   9|
|  Losing My Religion|              R.E.M.|alternative|        8

**In plain English:** "Within each genre, order tracks by popularity, highest first."

---

### 3.2 Find the Most Popular Track per Genre

Now we add a rank column using `rank().over(w)`, then filter to rank 1:

In [30]:
# Add rank column, look at one genre
df.withColumn("rank", rank().over(w))\
  .select("track_name","Artists", "track_genre", "popularity", "rank") \
  .filter(col("rank") == 1).show()

+-----------------------------------+--------------------+-------------+----------+----+
|                         track_name|             Artists|  track_genre|popularity|rank|
+-----------------------------------+--------------------+-------------+----------+----+
|                            Hold On|    Chord Overstreet|     acoustic|        82|   1|
|                     Atrévete-Te-Te|            Calle 13|     afrobeat|        75|   1|
|                    Sweater Weather|   The Neighbourhood|     alt-rock|        93|   1|
|                    Sweater Weather|   The Neighbourhood|  alternative|        93|   1|
|                         Apocalypse|Cigarettes After Sex|      ambient|        84|   1|
|                          KICK BACK|       Kenshi Yonezu|        anime|        83|   1|
|                         Doomswitch|    Make Them Suffer|  black-metal|        58|   1|
|                           Daylight|          Watchhouse|    bluegrass|        69|   1|
|                  Se

In [ ]:
# Filter to rank 1 = most popular per genre


**Notice:**
- Every row still exists until we filter (no collapse!)
- Each row has a rank within its genre
- Ties get the same rank (multiple rank 1s possible)

---

### 3.3 What About Top 3?

**Question:** Now find the top 3 most popular tracks in each genre.

## ✋ STOP - PREDICT FIRST

We have 114 genres. If I filter to `rank <= 3`:

1. Exactly how many rows should I get? (assume no ties)
   
   → My prediction: __342___________

2. Will I get exactly that many? Why or why not?
   
   → My prediction: _____________

In [35]:
# Filter to top 3 per genre
df.withColumn("rank", rank().over(w))\
  .select("track_name","Artists", "track_genre", "popularity", "rank") \
  .filter(col("rank").isin([1,2,3])).count()

411

### 3.4 Prediction: row_number() vs rank()

## ✋ STOP - PREDICT FIRST

If I use `row_number()` instead of `rank()`, and filter to `<= 3`:

1. Will I get MORE rows, FEWER rows, or SAME rows?
   
   → My prediction: _____________

2. Why?
   
   → My prediction: _____________

In [36]:
# Compare row_number() vs rank()
df.withColumn("rn", row_number().over(w))\
  .select("track_name","Artists", "track_genre", "popularity", "rn") \
  .filter(col("rn").isin([1,2,3])).count()

342

### Debrief: row_number vs rank vs dense_rank

For popularity values: 100, 100, 90, 80

| Function | Result | Use when... |
|----------|--------|--------------|
| `row_number()` | 1, 2, 3, 4 | You need exactly N rows per group |
| `rank()` | 1, 1, 3, 4 | Ties should share rank, gaps OK |
| `dense_rank()` | 1, 1, 2, 3 | Ties share rank, no gaps |

---

### 3.5 YOUR TURN: 60 Second Challenge ⏱️

**Modify the window to find the LEAST popular track in each genre.**

Hint: You only need to change one thing.

In [38]:
# YOUR TURN: Find the LEAST popular track in each genre
w = Window.partitionBy("track_genre").orderBy(("popularity"))

df.withColumn("rn", row_number().over(w))\
  .select("track_name","Artists", "track_genre", "popularity", "rn") \
  .filter(col("rn") == 1).show()

+--------------------+--------------------+-------------+----------+---+
|          track_name|             Artists|  track_genre|popularity| rn|
+--------------------+--------------------+-------------+----------+---+
|    93 Million Miles|          Jason Mraz|     acoustic|         0|  1|
|              Makoti|       Hugh Masekela|     afrobeat|         0|  1|
|          Nerve Flip|Red Hot Chili Pep...|     alt-rock|         0|  1|
|          Nerve Flip|Red Hot Chili Pep...|  alternative|         0|  1|
|  Written on the Sky|Max Richter;Johan...|      ambient|         0|  1|
|     WanteD! WanteD!|    Mrs. GREEN APPLE|        anime|         0|  1|
| Once upon the Cross|             Deicide|  black-metal|         0|  1|
|  Winter In My Heart|  The Avett Brothers|    bluegrass|         0|  1|
|This Girl's in Lo...|     Aretha Franklin|        blues|         0|  1|
|    Bring Me To Life|Bruno Martini;Dav...|       brazil|         0|  1|
|                  Go|The Chemical Brot...|    brea

---

## Part 4: lead() and lag() with Chart Data

These let you access values from other rows:
- `lag(col, n)` - value from n rows BEFORE
- `lead(col, n)` - value from n rows AFTER

**Example:** Let's use some Billboard-style chart data to see how songs move up and down the charts.

In [39]:
# Billboard-style chart data
chart_data = [
    # Week 1
    ("2024-01-06", "Lovin On Me", "Jack Harlow", 1),
    ("2024-01-06", "Cruel Summer", "Taylor Swift", 2),
    ("2024-01-06", "Water", "Tyla", 3),
    ("2024-01-06", "Stick Season", "Noah Kahan", 4),
    ("2024-01-06", "Agora Hills", "Doja Cat", 5),
    # Week 2 - positions changed
    ("2024-01-13", "Lovin On Me", "Jack Harlow", 1),
    ("2024-01-13", "Water", "Tyla", 2),        # up from 3
    ("2024-01-13", "Cruel Summer", "Taylor Swift", 3),  # down from 2
    ("2024-01-13", "Agora Hills", "Doja Cat", 4),  # up from 5
    ("2024-01-13", "Stick Season", "Noah Kahan", 5),  # down from 4
    # etc.
]

chart = spark.createDataFrame(chart_data, ["week", "song", "artist", "position"])
chart.show()

+----------+------------+------------+--------+
|      week|        song|      artist|position|
+----------+------------+------------+--------+
|2024-01-06| Lovin On Me| Jack Harlow|       1|
|2024-01-06|Cruel Summer|Taylor Swift|       2|
|2024-01-06|       Water|        Tyla|       3|
|2024-01-06|Stick Season|  Noah Kahan|       4|
|2024-01-06| Agora Hills|    Doja Cat|       5|
|2024-01-13| Lovin On Me| Jack Harlow|       1|
|2024-01-13|       Water|        Tyla|       2|
|2024-01-13|Cruel Summer|Taylor Swift|       3|
|2024-01-13| Agora Hills|    Doja Cat|       4|
|2024-01-13|Stick Season|  Noah Kahan|       5|
+----------+------------+------------+--------+



**Task: For each song, get last week's position and calculate how many spots they moved.**

In [46]:
# Window: for each song, order by week
w = Window.partitionBy("song").orderBy("week")

# add new column "last_week"
chart.withColumn("last_week", lag("position", 1).over(w)).orderBy("week","position") \
     .withColumn("change", col("position") - col("last_week")).show()


+----------+------------+------------+--------+---------+------+
|      week|        song|      artist|position|last_week|change|
+----------+------------+------------+--------+---------+------+
|2024-01-06| Lovin On Me| Jack Harlow|       1|     NULL|  NULL|
|2024-01-06|Cruel Summer|Taylor Swift|       2|     NULL|  NULL|
|2024-01-06|       Water|        Tyla|       3|     NULL|  NULL|
|2024-01-06|Stick Season|  Noah Kahan|       4|     NULL|  NULL|
|2024-01-06| Agora Hills|    Doja Cat|       5|     NULL|  NULL|
|2024-01-13| Lovin On Me| Jack Harlow|       1|        1|     0|
|2024-01-13|       Water|        Tyla|       2|        3|    -1|
|2024-01-13|Cruel Summer|Taylor Swift|       3|        2|     1|
|2024-01-13| Agora Hills|    Doja Cat|       4|        5|    -1|
|2024-01-13|Stick Season|  Noah Kahan|       5|        4|     1|
+----------+------------+------------+--------+---------+------+



**Notice:** 

- First week has NULL for last_week (no previous data)
- Positive change = moved UP the chart (lower position number is better)
- Water: went from 3 → 2, change = +1 (moved up 1 spot)
- Cruel Summer: went from 2 → 3, change = -1 (dropped 1 spot)

---

## Summary: Window Functions

```python
# 1. Define the window
w = Window.partitionBy("group_col").orderBy("sort_col")

# 2. Apply a function over it
df.withColumn("new_col", some_function().over(w))
```

**Common functions:**
- `rank()`, `dense_rank()`, `row_number()` - ranking
- `lag()`, `lead()` - access other rows
- `sum()`, `avg()` - running totals (with `rowsBetween`)

---

In [ ]:
spark.stop()
